## Lab4
### Topic: Advanced Net

### Завдання №3: Використання моделей з бібліотеки Hugging Face
---

### Частина A:
Взяти датасет для класифікації текстів (з другої лабораторної роботи) - в мене це Instagram відгуки.
Використайти модуль pipeline для запуску готових моделей (оберати мінімум дві моделі). Пропустити тексти через обрані моделі та зберегти результати. Провести оцінку якості класифікації: accuracy; classification report (precision, recall, F1).

Зробити висновки: Яка модель дала кращий результат? Провести порівняння з результатами з 2 лабораторної роботи - там застосовували традиційні ML методи (Naive Bayes, Logistic Regression).

### 1.1 Імпорт бібліотек

In [2]:
import pandas as pd
import numpy as np
import os
import json
import time

from datetime import datetime
from transformers import pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                            confusion_matrix, f1_score,
                            precision_score, recall_score)

from tqdm import tqdm # cute progress bars

# Приховуємо "зайві" попередження від transformers (deprecation warnings тощо)
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

np.random.seed(42)

### 1.2 Checkpoints

(як і в попередніх заваданнях перестраховка при відключеннях для перезапуску)

In [3]:
class CheckpointManager:
    """
    Сheckpoints для збереження та відновлення прогресу класифікації.
    """

    def __init__(self, checkpoint_dir='checkpoints'):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)

    def get_checkpoint_path(self, model_name):
        """Повертає шлях до файлу checkpoint для конкретної моделі."""
        safe_name = model_name.replace('/', '_').replace('-', '_')
        return os.path.join(self.checkpoint_dir, f'checkpoint_{safe_name}.json')

    def save_checkpoint(self, model_name, predictions, scores, last_index):
        """Зберігає поточний прогрес."""
        checkpoint_data = {
            'model_name': model_name,
            'predictions': predictions,
            'scores': scores,
            'last_index': last_index,
            'timestamp': datetime.now().isoformat()
        }

        path = self.get_checkpoint_path(model_name)
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(checkpoint_data, f, ensure_ascii=False)

        return path

    def load_checkpoint(self, model_name):
        """Завантажує збережений прогрес, якщо є."""
        path = self.get_checkpoint_path(model_name)

        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            print(f" Знайдено checkpoint від {data['timestamp']}")
            print(f"  Оброблено записів: {data['last_index']}")
            return data

        return None

    def clear_checkpoint(self, model_name):
        """Видаляє checkpoint після успішного завершення."""
        path = self.get_checkpoint_path(model_name)
        if os.path.exists(path):
            os.remove(path)
            print(f"Checkpoint видалено: {path}")

# Ініціалізація
checkpoint_manager = CheckpointManager()

### 1.3 Завантаження датасету

In [4]:
DATA_PATH = 'var_data/instagram.csv'
df = pd.read_csv(DATA_PATH)

print(f"Розмір: {df.shape[0]} записів, {df.shape[1]} колонок")
print(f"\nColumns: {df.columns.tolist()}")
df.head(5)

Розмір: 210542 записів, 2 колонок

Columns: ['Review', 'label']


,Review,label
0,"The app is good for connecting with friends, f...",NEGATIVE
1,"Used to be my favorite social media app, but ""...",NEGATIVE
2,Instagram is the best of all the social media....,POSITIVE
3,"I love this app.. but as of late, I have been ...",NEGATIVE
4,Used to be a great app but there are so many m...,NEGATIVE


Загальний обсяг: 210 542 записи. Датасет містить відгуки користувачів про додаток Instagram з Google Play Store за період вересень 2018 — липень 2023 року. Складається з двох колонок:
- `Review` — текст відгуку англійською мовою
- `label` — бінарна мітка сентименту (POSITIVE/NEGATIVE), для зручності зірковий рейтинг перевів за зірки за принципом(≥3 зірки → POSITIVE, ≤2 зірки → NEGATIVE) І все одно датасет є незбалансованим, бо переважають негативні відгуки (як можна бачити нижче).

In [5]:
print("Кількісний розподіл класів:")
class_counts = df['label'].value_counts()
print(class_counts)

print(f"\nPercentage Ratio:")
class_percentages = df['label'].value_counts(normalize=True) * 100
for label, pct in class_percentages.items():
    print(f"  {label}: {pct:.1f}%")

Кількісний розподіл класів:
label
NEGATIVE    147082
POSITIVE     63460
Name: count, dtype: int64

Percentage Ratio:
  NEGATIVE: 69.9%
  POSITIVE: 30.1%


Кількість негативних відгуків значно переважає. Це я враховуватиму далі при оцінці якості моделей.

In [6]:
# Проведемо чистку даних (обов'язково) і перейменуємо колонки (для зручності)
print(f"До очистки: {df.shape[0]} записів")

df = df.dropna(subset=['Review'])
df = df[df['Review'].str.strip() != '']

df = df.rename(columns={'Review': 'text', 'label': 'sentiment'})

print(f"Після очистки: {len(df)} записів")

До очистки: 210542 записів
Після очистки: 210542 записів


Як можна бачити, датасет в плані якості є хорошим, порожні поля в ньому відсутні) Але проблема з нерівномірним розподілом даних між класами залишається(

In [7]:
# Оскільки Hugging Face моделі працюють повільно на CPU, тому беремо оберемо певну підмножину з нашого датасету.

SAMPLE_SIZE = 10000 # Розмір вибірки

# Вибираємо екземпляри, зберігаючи пропорційне співвідношення між класами
df_sample = df.groupby('sentiment', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), int(SAMPLE_SIZE * len(x) / len(df))), 
                    random_state=42)
)

# Перемішуємо вибірку для уникнення впорядкування за класами
df_sample = df_sample.sample(frac=1, random_state=42).reset_index(drop=True)

texts = df_sample['text'].tolist()
labels = df_sample['sentiment'].tolist()

print(f"Розмір вибірки: {len(texts)}")
print(f"\nРозподіл класів у вибірці:")
sample_dist = pd.Series(labels).value_counts()
for label, count in sample_dist.items():
    print(f"  {label}: {count} ({count/len(labels)*100:.1f}%)")

Розмір вибірки: 9999

Розподіл класів у вибірці:
  NEGATIVE: 6985 (69.9%)
  POSITIVE: 3014 (30.1%)


### 1.4 Завантаження моделей Hugging Face

Використаємо дві моделі для sentiment analysis:

1. **DistilBERT (SST-2)** — швидка модель, тренована на фільмових рецензіях
2. **RoBERTa (Twitter)** — модель, тренована на Twitter даних (короткі тексти неофіційного стилю)

Обрав саме ці моделі, виходячи з того що мій датасет це відгуки. Застосування першої моделі було цікаво перевірити як модель з іншого домену (фільми) працює на відгуках про додаток. А друга - була навчена на трьох класах (positive/negative/neutral), то ж хочеться подивитися як вестиме себе на двох класах.

In [9]:
# Модель 1: DistilBERT
MODEL1_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

print(f"Завантаження моделі: {MODEL1_NAME}")

classifier1 = pipeline(
    "sentiment-analysis", 
    model=MODEL1_NAME, 
    device=-1  # CPU
)

print("DistilBERT завантажено!")

Завантаження моделі: distilbert-base-uncased-finetuned-sst-2-english


Device set to use cpu


DistilBERT завантажено!


In [10]:
# Модель 2: RoBERTa (Twitter)
MODEL2_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

print(f"Завантаження моделі: {MODEL2_NAME}")

classifier2 = pipeline(
    "sentiment-analysis", 
    model=MODEL2_NAME, 
    device=-1  # CPU
)

print("RoBERTa завантажено!")

Завантаження моделі: cardiffnlp/twitter-roberta-base-sentiment-latest


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


RoBERTa завантажено!


### 1.5 Власне класифікація текстів

In [11]:
def classify_with_checkpoints(classifier, texts, model_name, 
                            batch_size=16, 
                            checkpoint_every=500,
                            model_type='standard'):
    """
    Класифікує тексти з підтримкою checkpoints.

    Args:
        classifier: pipeline об'єкт Hugging Face
        texts: список текстів для класифікації
        model_name: назва моделі (для checkpoint)
        batch_size: розмір батчу (менший = повільніше, але стабільніше)
        checkpoint_every: зберігати checkpoint кожні N батчів
        model_type: 'standard' або 'twitter' (для маппінгу міток)

    Returns:
        predictions: список передбачень
        scores: список впевненості моделі
    """

    # Спробуємо завантажити існуючий checkpoint
    checkpoint = checkpoint_manager.load_checkpoint(model_name)

    if checkpoint:
        predictions = checkpoint['predictions']
        scores = checkpoint['scores']
        start_idx = checkpoint['last_index']
        print(f"Продовжуємо з індексу {start_idx}...")
    else:
        predictions = []
        scores = []
        start_idx = 0
        print("Починаємо з початку...")

    # Рахуємо кількість батчів
    total_batches = (len(texts) - start_idx + batch_size - 1) // batch_size
    batches_processed = 0

    # Основний цикл
    pbar = tqdm(range(start_idx, len(texts), batch_size), 
                desc="Класифікація",
                total=total_batches)

    for i in pbar:
        batch_texts = texts[i:i + batch_size]

        # Обрізаємо довгі тексти (ліміт моделі ~512 токенів)
        batch_texts = [t[:1000] if len(t) > 1000 else t for t in batch_texts]

        try:
            results = classifier(batch_texts, truncation=True, max_length=512)

            for result in results:
                label = result['label'].upper()
                score = result['score']

                # Маппінг міток
                if model_type == 'twitter':
                    # Twitter модель має 3 класи: positive, negative, neutral
                    if label == 'POSITIVE':
                        pred = 'POSITIVE'
                    elif label == 'NEGATIVE':
                        pred = 'NEGATIVE'
                    else:  # NEUTRAL -> NEGATIVE (консервативний підхід)
                        pred = 'NEGATIVE'
                else:
                    # Стандартна модель: POSITIVE/NEGATIVE
                    pred = label

                predictions.append(pred)
                scores.append(score)

        except Exception as e:
            print(f"\nError by обробці батчу {i}: {e}")
            # Зберігаємо checkpoint при помилці
            checkpoint_manager.save_checkpoint(model_name, predictions, scores, i)
            print(f"Checkpoint збережено. Reload cell to continue.")
            raise

        batches_processed += 1

        # Періодичне збереження checkpoint
        if batches_processed % checkpoint_every == 0:
            checkpoint_manager.save_checkpoint(
                model_name, predictions, scores, i + batch_size
            )
            pbar.set_postfix({'checkpoint': 'saved'})

    # Видаляємо checkpoint після успішного завершення класифікації
    checkpoint_manager.clear_checkpoint(model_name)

    return predictions, scores

In [16]:
print("=" * 60)
print("КЛАСИФІКАЦІЯ З DISTILBERT")
print("=" * 60)

start_time = time.time()

preds_distilbert, scores_distilbert = classify_with_checkpoints(
    classifier1, 
    texts, 
    MODEL1_NAME,
    batch_size=16,
    checkpoint_every=100
)

time_distilbert = time.time() - start_time
print(f"Час роботи {time_distilbert:.1f} секунд ({time_distilbert/60:.1f} хв)")

КЛАСИФІКАЦІЯ З DISTILBERT
Починаємо з початку...


Класифікація: 100%|██████████| 625/625 [07:05<00:00,  1.47it/s, checkpoint=saved]

Checkpoint видалено: checkpoints\checkpoint_distilbert_base_uncased_finetuned_sst_2_english.json
Час роботи 425.6 секунд (7.1 хв)


In [17]:
print("=" * 60)
print("КЛАСИФІКАЦІЯ З RоBERTa (TWITTER)")
print("=" * 60)

start_time = time.time()

preds_roberta, scores_roberta = classify_with_checkpoints(
    classifier2, 
    texts, 
    MODEL2_NAME,
    batch_size=16,
    checkpoint_every=100,
    model_type='twitter'
)

time_roberta = time.time() - start_time
print(f"\nЧас роботи {time_roberta:.1f} секунд ({time_roberta/60:.1f} хв)")

КЛАСИФІКАЦІЯ З RоBERTa (TWITTER)
Починаємо з початку...


Класифікація: 100%|██████████| 625/625 [13:31<00:00,  1.30s/it, checkpoint=saved]

Checkpoint видалено: checkpoints\checkpoint_cardiffnlp_twitter_roberta_base_sentiment_latest.json

Час роботи 811.5 секунд (13.5 хв)


Оскільки модель тренована не мною, я її тільки застосовував, то доцільно зберігти результати класифікації в CSV. (результати хороші і можуть бути застосовані до іншого датасету)

In [19]:
results_df = pd.DataFrame({
    'text': texts,
    'true_label': labels,
    'distilbert_pred': preds_distilbert,
    'distilbert_score': scores_distilbert,
    'roberta_pred': preds_roberta,
    'roberta_score': scores_roberta
})

results_df.to_csv('classification_results.csv', index=False)
print("Результати збережено в 'classification_results.csv'")

Результати збережено в 'classification_results.csv'


### 1.6 Оцінка якості класифікації

In [20]:
def evaluate_model(y_true, y_pred, model_name):
    """
    Комплексна оцінка якості моделі.
    """
    print(f"\n{'='*65}")
    print(f"РЕЗУЛЬТАТИ: {model_name}")
    print(f"{'='*65}")

    # Accuracy
    acc = accuracy_score(y_true, y_pred)
    print(f"\nAccuracy: {acc:.4f} ({acc*100:.2f}%)")

    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=2))

    # Confusion Matrix
    print("Confusion Matrix:")
    cm = confusion_matrix(y_true, y_pred, labels=['NEGATIVE', 'POSITIVE'])
    print(f"                 Predicted")
    print(f"                 NEG    POS")
    print(f"Actual NEG    [{cm[0,0]:5d}  {cm[0,1]:5d}]")
    print(f"       POS    [{cm[1,0]:5d}  {cm[1,1]:5d}]")

    fn = cm[1, 0]  # False Negatives (POSITIVE класифіковано як NEGATIVE)
    fp = cm[0, 1]  # False Positives (NEGATIVE класифіковано як POSITIVE)

    print(f"\nАналіз помилок:")
    print(f"  False Negatives (POS→NEG): {fn} ({fn/sum(cm[1,:])*100:.1f}% від POS)")
    print(f"  False Positives (NEG→POS): {fp} ({fp/sum(cm[0,:])*100:.1f}% від NEG)")

    return {
        'accuracy': acc,
        'weighted_f1': f1_score(y_true, y_pred, average='weighted'),
        'macro_f1': f1_score(y_true, y_pred, average='macro'),
        'precision_weighted': precision_score(y_true, y_pred, average='weighted'),
        'recall_weighted': recall_score(y_true, y_pred, average='weighted'),
        'confusion_matrix': cm
    }

In [21]:
# Оцінка DistilBERT
metrics_distilbert = evaluate_model(labels, preds_distilbert, "DistilBERT (SST-2)")


РЕЗУЛЬТАТИ: DistilBERT (SST-2)

Accuracy: 0.8458 (84.58%)

Classification Report:
              precision    recall  f1-score   support

    NEGATIVE       0.85      0.95      0.90      6985
    POSITIVE       0.83      0.61      0.71      3014

    accuracy                           0.85      9999
   macro avg       0.84      0.78      0.80      9999
weighted avg       0.84      0.85      0.84      9999

Confusion Matrix:
                 Predicted
                 NEG    POS
Actual NEG    [ 6607    378]
       POS    [ 1164   1850]

Аналіз помилок:
  False Negatives (POS→NEG): 1164 (38.6% від POS)
  False Positives (NEG→POS): 378 (5.4% від NEG)


In [22]:
# Оцінка RoBERTa
metrics_roberta = evaluate_model(labels, preds_roberta, "RoBERTa (Twitter)")


РЕЗУЛЬТАТИ: RoBERTa (Twitter)

Accuracy: 0.8708 (87.08%)

Classification Report:
              precision    recall  f1-score   support

    NEGATIVE       0.87      0.97      0.91      6985
    POSITIVE       0.89      0.65      0.75      3014

    accuracy                           0.87      9999
   macro avg       0.88      0.81      0.83      9999
weighted avg       0.87      0.87      0.86      9999

Confusion Matrix:
                 Predicted
                 NEG    POS
Actual NEG    [ 6743    242]
       POS    [ 1050   1964]

Аналіз помилок:
  False Negatives (POS→NEG): 1050 (34.8% від POS)
  False Positives (NEG→POS): 242 (3.5% від NEG)


**Аналіз Hugging Face моделей:**

RoBERTa (Twitter) показала кращі результати порівняно з DistilBERT:
- Вища accuracy (87.08% vs 84.58%)
- Кращий weighted F1 (F score) (0.86 vs 0.84)
- Вищий recall для POSITIVE класу (65% vs 61%)
- Менше неправильних спрацьовувань (False Negatives) (1050 vs 1164)

Це можна пояснити тим, що RoBERTa тренувалася на Twitter даних — де здебільшого короткий неформальний текст, що ближче до стилю Instagram відгуків. DistilBERT, натренована на фільмових рецензіях (SST-2), працює з іншим доменом — тексти там довші та більш формальні.

Обидві моделі демонструють типову поведінку при дисбалансі класів: краще розпізнають NEGATIVE (recall ~95-97%), гірше — POSITIVE (recall ~61-65%).

### 1.7 Порівняння усіх моделей разом (Hugging Face + TfidfVectorizer/ML).

**Зведена таблиця результатів (Hugging Face + традиційні ML методи з лаби 2):**

| Модель                        | Accuracy   | Weighted F1 | Recall (POS) | Precision (POS) | Час    |
| ----------------------------- | ---------- | ----------- | ------------ | --------------- | ------ |
| Naive Bayes (TF-IDF)          | 85.00%     | 0.84        | 59%          | 86%             | <1 хв  |
| Logistic Regression (TF-IDF)  | 86.00%     | 0.86        | 68%          | 84%             | <2 хв  |
| DistilBERT (HuggingFace)      | 84.58%     | 0.84        | 61%          | 83%             | ~7 хв  |
| RoBERTa Twitter (HuggingFace) | **87.08%** | **0.86**    | 65%          | **89%**         | ~13 хв |

Який з цього висновок:

1. **Найкраща модель за Accuracy та F1:** RoBERTa (Twitter) — 87.08%
2. **Найкраща модель за Recall (POS):** Logistic Regression — 68%
3. **Найкраща модель за Precision (POS):** RoBERTa — 89%

**Ключові спостереження:**

- RoBERTa перемагає за загальною точністю, але Logistic Regression краще "ловить" позитивні відгуки (вищий recall)
- Традиційні ML методи працюють у 10-15 разів швидше (що є логічним, бо вони математично простіше) на CPU
- DistilBERT показала найгірший результат — домен тренування (фільмові рецензії) занадто відрізняється від характеру написання коротких нестандартизованих відгуків
- Дисбаланс класів (70/30) впливає на всі моделі однаково — всі краще розпізнають саме NEGATIVE


---

### Частина Б: Україномовні моделі в Hugging Face

Наступним завданням я обрав варіант з україномовними моделями для різних NLP задач:
- Zero-shot Classification
- Summarization
- Translation

### 2.1 Zero-shot Classification (Класифікація без навчання)

Zero-shot класифікація це задача, яка дозволяє класифікувати тексти за категоріями, на яких модель не навчалася явно. Для цього завдання я обрав сучасну мультимовну модель mDeBERTa, що підтримує українську мову.

In [27]:
print("Завантаження моделі для zero-shot класифікації...")

zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
    device=-1
)

print("Модель завантажено!")

Завантаження моделі для zero-shot класифікації...


Device set to use cpu


Модель завантажено!


Подаю моделі на вхід довільно придумані речення, і прошу класифікувати їх відносно заданих категорій:

In [29]:
ukrainian_texts = [
    "Збірна України з футболу виграла важливий матч у відборі на Євро 2012.",
    "Вчені з Інституту молекулярної біології відкрили новий метод лікування раку.",
    "Курс гривні залишається стабільним попри економічно складну ситуацію через війну.",
    "Новий український фільм отримав нагороду на Берлінському кінофестивалі.",
    "Верховна Рада України прийняла закон про підтримку малого бізнесу.",
    "Комптютерні технології стрімко розвиваються, відкриваючи нові можливості для інновацій."
]

categories = ["спорт", "наука", "економіка", "культура", "політика"]

print("=" * 70)
print("ZERO-SHOT КЛАСИФІКАЦІЯ УКРАЇНСЬКИХ ТЕКСТІВ")
print("=" * 70)
print(f"Всі категорії: {', '.join(categories)}")
print()

for i, text in enumerate(ukrainian_texts, 1):
    result = zero_shot_classifier(text, categories)

    print(f"\n[{i}] {text}")
    print(f"    → Категорія: {result['labels'][0]} (впевненість: {result['scores'][0]:.1%})")

    # Показуємо топ-3 ймовірні категорії
    print(f"    Всі scores: ", end="")
    for label, score in zip(result['labels'][:3], result['scores'][:3]):
        print(f"{label}: {score:.1%}  ", end="")
    print()

ZERO-SHOT КЛАСИФІКАЦІЯ УКРАЇНСЬКИХ ТЕКСТІВ
Всі категорії: спорт, наука, економіка, культура, політика


[1] Збірна України з футболу виграла важливий матч у відборі на Євро 2012.
    → Категорія: спорт (впевненість: 97.9%)
    Всі scores: спорт: 97.9%  політика: 0.9%  культура: 0.9%  

[2] Вчені з Інституту молекулярної біології відкрили новий метод лікування раку.
    → Категорія: наука (впевненість: 97.3%)
    Всі scores: наука: 97.3%  культура: 1.2%  економіка: 0.7%  

[3] Курс гривні залишається стабільним попри економічно складну ситуацію через війну.
    → Категорія: економіка (впевненість: 73.6%)
    Всі scores: економіка: 73.6%  політика: 16.6%  наука: 5.5%  

[4] Новий український фільм отримав нагороду на Берлінському кінофестивалі.
    → Категорія: культура (впевненість: 63.8%)
    Всі scores: культура: 63.8%  наука: 14.5%  політика: 13.4%  

[5] Верховна Рада України прийняла закон про підтримку малого бізнесу.
    → Категорія: політика (впевненість: 66.5%)
    Всі scores: 

Також ця модель вміє проводити sentiment analysis і визначати "характер" (настрій) тексту.

In [31]:
print("\n" + "=" * 70)
print("ZERO-SHOT: визначення емоційного забарвлення текстів")
print("=" * 70)

sentiment_labels = ["позитивний", "негативний", "нейтральний"]

test_reviews = [
    "Чудовий додаток! Користуюсь щодня і дуже задоволений.",
    "На трієчку. Для не частого користування підійде.",
    "Жахливе оновлення, розробники ідійоти. Видаляю!",
    "Ставлю п'ять зірок, все супер!",
    "Це найгірший сервіс, з яким я коли-небудь мав справу.",
    "Додаток працює нормально, нічого особливого."
]

for text in test_reviews:
    result = zero_shot_classifier(text, sentiment_labels)
    print(f"\n\"{text}\"")
    print(f"  → {result['labels'][0]} ({result['scores'][0]:.1%})")


ZERO-SHOT: визначення емоційного забарвлення текстів

"Чудовий додаток! Користуюсь щодня і дуже задоволений."
  → позитивний (96.2%)

"На трієчку. Для не частого користування підійде."
  → позитивний (53.6%)

"Жахливе оновлення, розробники ідійоти. Видаляю!"
  → негативний (66.6%)

"Ставлю п'ять зірок, все супер!"
  → позитивний (96.7%)

"Це найгірший сервіс, з яким я коли-небудь мав справу."
  → негативний (93.3%)

"Додаток працює нормально, нічого особливого."
  → нейтральний (78.4%)


Проміжний підсумок по Zero-shot Classification:

Модель mDeBERTa демонструє нам вражаючі результати на українських текстах без будь-якого додаткового навчання. Це її головна перевага (можливість класифікувати тексти за довільними заданими категоріями без збору тренувальних даних та донавчання).

**Класифікація за категоріями:**
- Однозначні випадки (спорт, наука) класифікуються з впевненістю 97-98%;
- Тексти з перетином категорій (економіка/політика, культура/наука) мають нижчу впевненість (63-74%), що логічно — наприклад, закон про бізнес стосується як політики, так і економіки.

**Визначення емоційного забарвлення тексту:**
- Яскраво виражені емоції розпізнаються з високою точністю (93-97%)
- Нейтральні тексти коректно класифікуються (78%)
- Неоднозначний відгук "На трієчку" отримав лише 53.6% позитивного — модель "не впевнена", що є цілком адекватною поведінкою


### 2.2 Summarization (Резюмування тексту)

Для резюмування (виділення основного посилу тексту) я використав мультимовну модель mT5.

In [32]:
print("Завантаження моделі для резюмування (mT5)...")

summarizer = pipeline(
    "summarization",
    model="csebuetnlp/mT5_multilingual_XLSum",
    device=-1
)

print("Модель завантажено!")

Завантаження моделі для резюмування (mT5)...


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Device set to use cpu


Модель завантажено!


In [38]:
# Приклад 1: Новина про технології
print("=" * 70)
print("РЕЗЮМУВАННЯ УКРАЇНСЬКОГО ТЕКСТУ")
print("=" * 70)

text_tech = """
Volkswagen ID.Unyx — новий електрокросовер Volkswagen, що нещодавно став найпопулярнішою новою моделлю 
на українському ринку. Це показує: навіть у складних економічних умовах попит на «зелені» авто стрімко 
зростає, і багато покупців готові обирати сучасні технологічні рішення. Проте Volkswagen зараз перебуває 
під тиском через глобальний дефіцит мікросхем. Концерн оголосив, що призупиняє виробництво кількох ключових 
моделей — серед них легендарний Volkswagen Golf та інші популярні моделі. Це змушує фірму переосмислити свої 
виробничі плани та шукати нові шляхи адаптації. За словами керівництва, Volkswagen вже інвестує мільярди євро 
у цифрові технології, автоматизацію та оптимізацію ланцюгів постачань. Це має допомогти зменшити залежність 
від критичного обладнання та зробити бізнес більш стійким до зовнішніх шоків. Аналітики прогнозують, що у 
короткостроковій перспективі підприємство може вийти із кризи з фокусом на електрокари. Якщо ці зусилля 
принесуть успіх — бренд може перезапустити себе, адаптувавшись до нової ери.
"""

print(f"\nОригінал ({len(text_tech.split())} слів):")
print(text_tech.strip())

summary = summarizer(text_tech, max_length=80, min_length=30, do_sample=False)

print(f"\n{'─'*40}")
print(f"Резюме:")
print(summary[0]['summary_text'])

РЕЗЮМУВАННЯ УКРАЇНСЬКОГО ТЕКСТУ

Оригінал (137 слів):
Volkswagen ID.Unyx — новий електрокросовер Volkswagen, що нещодавно став найпопулярнішою новою моделлю 
на українському ринку. Це показує: навіть у складних економічних умовах попит на «зелені» авто стрімко 
зростає, і багато покупців готові обирати сучасні технологічні рішення. Проте Volkswagen зараз перебуває 
під тиском через глобальний дефіцит мікросхем. Концерн оголосив, що призупиняє виробництво кількох ключових 
моделей — серед них легендарний Volkswagen Golf та інші популярні моделі. Це змушує фірму переосмислити свої 
виробничі плани та шукати нові шляхи адаптації. За словами керівництва, Volkswagen вже інвестує мільярди євро 
у цифрові технології, автоматизацію та оптимізацію ланцюгів постачань. Це має допомогти зменшити залежність 
від критичного обладнання та зробити бізнес більш стійким до зовнішніх шоків. Аналітики прогнозують, що у 
короткостроковій перспективі підприємство може вийти із кризи з фокусом на електрокари

In [36]:
# Приклад 2: Наукова новина
text_science = """
Новітні флагманські смартфони корейської компанії Samsung стрімко змінюють сучасний світ мобільних технологій. 
Пристрої, такі як Galaxy S25 Ultra, демонструють вражаючі можливості, особливо у сфері інтеграції 
Штучного Інтелекту (Galaxy AI), фотографії та продуктивності. Мобільні оператори та 
рітейлери активно впроваджують ці нові пристрої у свої стартапи. За прогнозами 
аналітиків ринку, частка преміальних смартфонів Samsung в Україні прогнозовано зросте на 30% 
протягом наступного року, завдяки попиту на AI-функціонал та підвищенню якості камер. Samsung 
продовжує задавати високі стандарти на ринку, перетворюючи флагманські смартфони не просто на засоби 
зв'язку, а на потужні персональні комп'ютери та інструменти творчості.
"""

print(f"\nОригінал ({len(text_science.split())} слів):")
print(text_science.strip())

summary2 = summarizer(text_science, max_length=60, min_length=15, do_sample=False)

print(f"\n{'─'*40}")
print(f"Резюме:")
print(summary2[0]['summary_text'])


Оригінал (92 слів):
Новітні флагманські смартфони корейської компанії Samsung стрімко змінюють сучасний світ мобільних технологій. 
Пристрої, такі як Galaxy S25 Ultra, демонструють вражаючі можливості, особливо у сфері інтеграції 
Штучного Інтелекту (Galaxy AI), фотографії та продуктивності. Мобільні оператори та 
рітейлери активно впроваджують ці нові пристрої у свої стартапи. За прогнозами 
аналітиків ринку, частка преміальних смартфонів Samsung в Україні прогнозовано зросте на 30% 
протягом наступного року, завдяки попиту на AI-функціонал та підвищенню якості камер. Samsung 
продовжує задавати високі стандарти на ринку, перетворюючи флагманські смартфони не просто на засоби 
зв'язку, а на потужні персональні комп'ютери та інструменти творчості.

────────────────────────────────────────
Резюме:
Компанія Samsung представила свої нові смартфони Galaxy S25 Ultra.


Короткі висновки по задачі Summarization:

Модель mT5 успішно генерує резюме українських текстів, проте має певні особливості:

**Переваги:**
- Майже коректно виділяє головну думку тексту
- Генерує граматично правильні українські речення (це мене приємно здивувало!)
- Зберігає ключові сутності (назви компаній, продуктів)

**Недоліки:**
- Резюме інколи занадто спрощене — втрачаються важливі деталі (наприклад, про інвестиції Volkswagen і їх вплив) - для цього треба експерементувати з `max_length` та `min_length`
- Або ж модель може обрати лише один аспект, ігноруючи інші (текст про Samsung мав багато деталей про AI та прогнози, але резюме згадує тільки презентацію)

### 2.3 Translation (Переклад)

Helsinki-NLP має спеціалізовані моделі для перекладу між мовами, наприклад українська ↔ англійська.

In [39]:
print("Завантаження моделей для перекладу...")

# UK → EN
translator_uk_en = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-uk-en",
    device=-1
)
print("UK→EN завантажено")

# EN → UK
translator_en_uk = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-en-uk",
    device=-1
)
print("EN→UK завантажено")

Завантаження моделей для перекладу...


Device set to use cpu


UK→EN завантажено


Device set to use cpu


EN→UK завантажено


In [41]:
print("=" * 70)
print("ПЕРЕКЛАД: УКРАЇНСЬКА → АНГЛІЙСЬКА")
print("=" * 70)

ukrainian_sentences = [
    "Україна — європейська держава з багатою історією та культурою.",
    "Машинне навчання допомагає вирішувати складні задачі реального світу.",
    "Київ є столицею України та одним з найстаріших міст Європи.",
    "Нейронні мережі здатні розпізнавати образи та генерувати тексти.",
    "Софіє, дякую Вам за допомогу, Ваш вклад є неоціненним!"
]

for text in ukrainian_sentences:
    translation = translator_uk_en(text)
    print(f"\n🇺🇦 {text}")
    print(f"🇬🇧 {translation[0]['translation_text']}")

ПЕРЕКЛАД: УКРАЇНСЬКА → АНГЛІЙСЬКА

🇺🇦 Україна — європейська держава з багатою історією та культурою.
🇬🇧 Ukraine is a European state with a rich history and culture.

🇺🇦 Машинне навчання допомагає вирішувати складні задачі реального світу.
🇬🇧 Machine learning helps solve complex problems in the real world.

🇺🇦 Київ є столицею України та одним з найстаріших міст Європи.
🇬🇧 Kiev is the capital of Ukraine and one of the oldest cities in Europe.

🇺🇦 Нейронні мережі здатні розпізнавати образи та генерувати тексти.
🇬🇧 The neural networks are able to recognize images and generate text.

🇺🇦 Софіє, дякую Вам за допомогу, Ваш вклад є неоціненним!
🇬🇧 Sofia, thank you for your help, your contribution is invaluable!


In [42]:
print("\n" + "=" * 70)
print("ПЕРЕКЛАД: АНГЛІЙСЬКА → УКРАЇНСЬКА")
print("=" * 70)

english_sentences = [
    "Machine learning is a subset of artificial intelligence.",
    "Neural networks can learn complex patterns from data.",
    "Deep learning has revolutionized computer vision and NLP.",
    "Transformers are the backbone of modern language models.",
    "Python is the most popular programming language for AI research."
]

for text in english_sentences:
    translation = translator_en_uk(text)
    print(f"\n🇬🇧 {text}")
    print(f"🇺🇦 {translation[0]['translation_text']}")


ПЕРЕКЛАД: АНГЛІЙСЬКА → УКРАЇНСЬКА

🇬🇧 Machine learning is a subset of artificial intelligence.
🇺🇦 Вивчення машин - це частина штучного інтелекту.

🇬🇧 Neural networks can learn complex patterns from data.
🇺🇦 Нервові мережі можуть вивчити складні візерунки на основі даних.

🇬🇧 Deep learning has revolutionized computer vision and NLP.
🇺🇦 Глибоке навчання докорінно змінило комп'ютерний зір і NLP.

🇬🇧 Transformers are the backbone of modern language models.
🇺🇦 Трансформатори - це основа сучасних мовних моделей.

🇬🇧 Python is the most popular programming language for AI research.
🇺🇦 Python є найпопулярнішою мовою програмування для досліджень ШІ.


Ще було цікаво зробити круговий переклад (UK → EN → UK), і подивитися наскільки текст в кінці буде відрізнятися від початкового

In [45]:
print("\n" + "=" * 70)
print("КРУГОВИЙ ПЕРЕКЛАД (UK → EN → UK)")
print("=" * 70)

original = "Я готовий зустріти свого Творця. Інша справа, чи готовий Творець до такого важкого випробування, як зустріч зі мною. - Вінстон Черчилль"

# UK → EN
to_english = translator_uk_en(original)[0]['translation_text']

# EN → UK
back_to_ukrainian = translator_en_uk(to_english)[0]['translation_text']

print(f"\nОригінал:        {original}")
print(f"→ Англійська:    {to_english}")
print(f"→ Назад в укр:   {back_to_ukrainian}")


КРУГОВИЙ ПЕРЕКЛАД (UK → EN → UK)

Оригінал:        Я готовий зустріти свого Творця. Інша справа, чи готовий Творець до такого важкого випробування, як зустріч зі мною. - Вінстон Черчилль
→ Англійська:    I'm ready to meet my Creator, and it's another thing whether the Creator is ready for such a difficult test as meeting me. - Winston Churchill
→ Назад в укр:   Я готовий зустрітися зі своїм Творцем, і зовсім інша річ, чи готовий Творець до такого важкого випробування, як зустріч зі мною (Вінстон Черчілль).


Висновок по Translation:

**UK → EN (українська → англійська):**
- Переклад загалом якісний та природний (по якості точно не Google Translate, a Deepl як мінімум)
- Технічні терміни перекладаються коректно ("Нейронні мережі" → "Neural networks")
- Зберігається структура та сенс речень

**EN → UK (англійська → українська):**
- Виявлено проблеми з технічною термінологією:
  - "Machine learning" → "Вивчення машин" (хоча правильно: "Машинне навчання")
  - "Neural networks" → "Нервові мережі" (а правильно: "Нейронні мережі")
  - "patterns" → "візерунки" (в контексті ML краще: дослівно перекласти як "патерни" або ж "закономірності")
- Модель можна зрозуміти, вона навчалася на загальновживаних словах і не адаптована до IT-термінології

**Круговий переклад:**

Сенс повідомлення зберігається повністю, хоча і присутні невеликі зміни у формулюванні ("Інша справа" → "зовсім інша річ") та змінена пунктуація ("- Вінстон Черчилль" → "(Вінстон Черчілль)"), що в загальному на розуміння не впливає.

Як підсумок, можна сказати що для технічних текстів краще використовувати все спеціалізовані моделі або post-editing (додаткова перевірка людиною-лінгвістом тексту, який був перекладений певною моделлю).

---
## Загальні висновки

**Частина A: Класифікація Instagram відгуків**

Порівняно 4 моделі для sentiment analysis: 2 традиційні ML (Naive Bayes, Logistic Regression з TF-IDF) та 2 Hugging Face (DistilBERT, RoBERTa).

Ключові результати:
- **RoBERTa (Twitter)** показала найвищу accuracy (87.08%) та precision для POSITIVE класу (89%)
- **Logistic Regression** має найкращий recall для POSITIVE (68%) — краще "ловить" позитивні відгуки
- **DistilBERT** поступилася навіть традиційним методам через невідповідність домену (фільмові рецензії vs короткі відгуки про додаток)
- Transformers (DistilBERT, RoBERTa) працюють в 10-15 разів повільніше на CPU, бо багато параметрів і нема розпараленення, але не потребують feature engineering (ручного створення ознак для класифікації)

**Частина Б: Україномовні моделі**

Попрацював з трьома NLP моделями, які підтримують українську мову:

1. **Zero-shot Classification (mDeBERTa):**
   - Класифікація текстів за довільними категоріями без навчання
   - Висока точність на однозначних прикладах (97-98%)
   - Працює як для тематичної класифікації, так і для sentiment analysis

2. **Summarization (mT5):**
   - Генерує граматично правильні резюме українською
   - Виділяє головну думку, але може втрачати деякі (+- важливі) нюанси

3. **Translation (Helsinki-NLP):**
   - UK→EN: якісний переклад
   - EN→UK: проблеми з технічною термінологією

**Загальний висновок:**
Hugging Face надає потужні інструменти для NLP задач, які можна застосувати без додаткового дотреновування і без створення власних моделей. Однак для задач які виходитимуть в прод, важливо враховувати відповідність домену тренувальних даних та специфіку задачі (технічна термінологія, дисбаланс класів тощо).